# Week 85, Day 2: Review: web and security

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w085d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
raw = "POST /notes?draft=1 HTTP/1.1\r\nHost: example.com\r\nContent-Type: application/json\r\nContent-Length: 15\r\n\r\n{\"text\": \"hi\"}"
head, body = raw.split("\r\n\r\n", 1)
lines = head.split("\r\n")
method, target, version = lines[0].split(" ")
headers = dict(line.split(": ", 1) for line in lines[1:])
print(method, target.split("?"), version)
print(headers["Content-Type"], len(body), body)

In [ ]:
import json
from urllib.parse import parse_qs, urlsplit

parts = urlsplit("/search?q=lamp&page=2&tag=home&tag=sale")
print(parts.path, parse_qs(parts.query))

body = json.dumps({"name": "café"}, ensure_ascii=False)
print(len(body), len(body.encode("utf-8")))
response = (f"HTTP/1.1 201 Created\r\nContent-Type: application/json\r\n"
            f"Content-Length: {len(body.encode())}\r\nLocation: /items/7\r\n\r\n{body}")
print(response.split("\r\n")[0], "|", response.split("\r\n")[2])

In [ ]:
import base64
import hashlib
import hmac
import json

SECRET = b"demo-" + b"secret"           # in real code: from the environment

def sign(payload):
    data = base64.urlsafe_b64encode(json.dumps(payload).encode()).decode().rstrip("=")
    sig = hmac.new(SECRET, data.encode(), hashlib.sha256).hexdigest()
    return f"{data}.{sig}"

token = sign({"user": "ada", "exp": 1000})
data, sig = token.split(".")
good = hmac.new(SECRET, data.encode(), hashlib.sha256).hexdigest()
print(len(sig), hmac.compare_digest(sig, good))
tampered = base64.urlsafe_b64encode(json.dumps({"user": "admin", "exp": 1000}).encode()).decode().rstrip("=")
print(hmac.compare_digest(sig, hmac.new(SECRET, tampered.encode(), hashlib.sha256).hexdigest()))

In [ ]:
import html
from pathlib import Path

comment = '<script>steal(document.cookie)</script> Nice post!'
print(html.escape(comment))

base = Path("/srv/uploads").resolve()
for name in ["report.pdf", "../../etc/passwd", "photos/cat.jpg"]:
    target = (base / name).resolve()
    print(name, "->", "allowed" if target.is_relative_to(base) else "blocked")

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="notes")
sub = parser.add_subparsers(dest="command", required=True)
add = sub.add_parser("add")
add.add_argument("text")
add.add_argument("--tag", action="append", default=[])
show = sub.add_parser("list")
show.add_argument("--limit", type=int, default=10)
print(parser.parse_args(["add", "buy milk", "--tag", "home", "--tag", "today"]))
print(parser.parse_args(["list", "--limit", "3"]))

> **Key idea.** HTTP messages are text: a start line, case-insensitive headers, a blank line and a body whose `Content-Length` counts bytes. Choose status codes precisely (201 with `Location`, 204, 400, 401 versus 403, 404, 405 with `Allow`, 409, 422, 429, 500). GET is safe; PUT and DELETE are idempotent; POST is neither. REST uses nouns for resources, methods for actions and consistent JSON errors; a framework is a router plus error handling. Signed tokens are `payload.signature` with an HMAC, checked with `compare_digest` and an expiry. Defend against injection with parameters, XSS with escaping, traversal by resolving and checking paths; keep secrets in the environment. Command-line tools take `argv`, use argparse and return exit codes.

### Exercise: Parsing a request

Write `parse_request(raw)` for the text of an HTTP request (lines ending in `\r\n`, a blank line before the body). Return a dict with `method`, `path` (without the query string), `query` (a dict from each name to its **last** value), `version`, `headers` (a dict with **lower-cased** names) and `body` (a string, possibly empty). Raise `ValueError("bad request line")` if the first line doesn't have exactly three parts.

In [ ]:
from urllib.parse import parse_qs, urlsplit

def parse_request(raw):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_request():\n    \"line, query, headers, body\"\n    raw = \"POST /notes?draft=1&tag=a&tag=b HTTP/1.1\\r\\nHost: example.com\\r\\nContent-TYPE: application/json\\r\\n\\r\\n{\\\"text\\\": \\\"hi\\\"}\"\n    r = parse_request(raw)\n    assert (r[\"method\"], r[\"path\"], r[\"version\"]) == (\"POST\", \"/notes\", \"HTTP/1.1\")\n    assert r[\"query\"] == {\"draft\": \"1\", \"tag\": \"b\"}\n    assert r[\"headers\"] == {\"host\": \"example.com\", \"content-type\": \"application/json\"}\n    assert r[\"body\"] == '{\"text\": \"hi\"}'\n    g = parse_request(\"GET / HTTP/1.1\\r\\nHost: x\\r\\n\\r\\n\")\n    assert g[\"path\"] == \"/\" and g[\"query\"] == {} and g[\"body\"] == \"\"\n    try:\n        parse_request(\"GET /\\r\\n\\r\\n\")\n    except ValueError as e:\n        assert str(e) == \"bad request line\"\n    else:\n        raise AssertionError(\"should fail\")")

### Exercise: Building a response

Write `build_response(status, data=None, headers=None)`: the text of an HTTP/1.1 response with the reason phrase from `REASONS` (given). With `data`, the body is `json.dumps(data, ensure_ascii=False)` and the headers include `Content-Type: application/json` and a `Content-Length` counting the body's UTF-8 **bytes**; with no data (as for 204), there's no body and `Content-Length: 0`. Extra `headers` come after those, in the order given. Lines end in `\r\n` and a blank line separates the headers from the body.

In [ ]:
import json

REASONS = {200: "OK", 201: "Created", 204: "No Content", 400: "Bad Request", 401: "Unauthorized",
           403: "Forbidden", 404: "Not Found", 405: "Method Not Allowed", 409: "Conflict",
           422: "Unprocessable Content", 429: "Too Many Requests", 500: "Internal Server Error"}

def build_response(status, data=None, headers=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_response():\n    \"status line, bytes, blank line\"\n    r = build_response(201, {\"name\": \"caf\u00e9\"}, {\"Location\": \"/items/7\"})\n    assert r == ('HTTP/1.1 201 Created\\r\\nContent-Type: application/json\\r\\nContent-Length: 17\\r\\n'\n                 'Location: /items/7\\r\\n\\r\\n{\"name\": \"caf\u00e9\"}')\n    assert build_response(204) == \"HTTP/1.1 204 No Content\\r\\nContent-Length: 0\\r\\n\\r\\n\"\n    assert build_response(405, {\"error\": \"method not allowed\"}, {\"Allow\": \"GET, POST\"}).startswith(\"HTTP/1.1 405 Method Not Allowed\\r\\n\")")

### Exercise: A router

Write a class `Router` with `add(method, pattern, handler)`, where a pattern's parts may be `<name>` (any one path segment) or `<int:name>` (digits only, converted to `int`). `match(method, path)` returns `(200, handler, params)` for a match; `(405, None, {"allow": "GET, POST"})` (the sorted methods registered for paths matching the pattern) if the path matches but not the method; and `(404, None, {})` otherwise. Patterns are tried in the order they were added.

In [ ]:
class Router:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_router():\n    \"patterns, conversions, 404 and 405\"\n    r = Router()\n    r.add(\"GET\", \"/notes\", \"list\")\n    r.add(\"POST\", \"/notes\", \"create\")\n    r.add(\"GET\", \"/notes/<int:id>\", \"show\")\n    r.add(\"DELETE\", \"/notes/<int:id>\", \"delete\")\n    r.add(\"GET\", \"/users/<name>/notes\", \"by_user\")\n    assert r.match(\"GET\", \"/notes\") == (200, \"list\", {})\n    assert r.match(\"GET\", \"/notes/42\") == (200, \"show\", {\"id\": 42})\n    assert r.match(\"GET\", \"/users/ada/notes\") == (200, \"by_user\", {\"name\": \"ada\"})\n    assert r.match(\"PUT\", \"/notes/42\") == (405, None, {\"allow\": \"DELETE, GET\"})\n    assert r.match(\"PATCH\", \"/notes\") == (405, None, {\"allow\": \"GET, POST\"})\n    assert r.match(\"GET\", \"/notes/abc\") == (404, None, {})\n    assert r.match(\"GET\", \"/notes/42/extra\") == (404, None, {})\n    assert r.match(\"GET\", \"/\") == (404, None, {})")

### Exercise: Signed tokens with expiry

Write `make_token(user, secret, now, ttl)`: a token `DATA.SIG`, where `DATA` is the URL-safe base64 (with `=` padding removed) of the JSON `{"user": user, "exp": now + ttl}` and `SIG` the hex HMAC-SHA256 of `DATA` with `secret` (bytes). Then `read_token(token, secret, now)`: return the user, or raise `ValueError` with `"malformed token"` (not two parts, or a payload that can't be decoded), `"bad signature"` (checked with `hmac.compare_digest` **before** decoding) or `"expired"` (`now >= exp`).

In [ ]:
import base64
import hashlib
import hmac
import json

def make_token(user, secret, now, ttl):
    ___

def read_token(token, secret, now):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import base64\nimport json\n\nSECRET = b\"test-\" + b\"only-\" + b\"key\"\n\ndef _error(token, now):\n    try:\n        read_token(token, SECRET, now)\n    except ValueError as e:\n        return str(e)\n    return None\n\ndef test_tokens():\n    \"sign, check, expire\"\n    t = make_token(\"ada\", SECRET, 1000, 300)\n    data, sig = t.split(\".\")\n    assert \"=\" not in data and len(sig) == 64\n    padded = data + \"=\" * (-len(data) % 4)\n    assert json.loads(base64.urlsafe_b64decode(padded)) == {\"user\": \"ada\", \"exp\": 1300}\n    assert read_token(t, SECRET, 1299) == \"ada\"\n    assert _error(t, 1300) == \"expired\"\n    forged = base64.urlsafe_b64encode(json.dumps({\"user\": \"admin\", \"exp\": 9999}).encode()).decode().rstrip(\"=\")\n    assert _error(f\"{forged}.{sig}\", 1000) == \"bad signature\"\n    assert _error(t, 1000) is None and _error(\"no-dot-here\", 1000) == \"malformed token\"\n    assert _error(make_token(\"bo\", SECRET, 0, 10), 1) is None\n    other = make_token(\"bo\", b\"another-\" + b\"key\", 0, 10)\n    assert _error(other, 1) == \"bad signature\"\n    assert \"compare_digest\" in source")

### Exercise: Three defences

Write `safe_path(base, name)`: the resolved `Path` for `name` inside the folder `base`, or `PermissionError("outside the folder")` if it would lead anywhere else (including through `..` or an absolute path). `render_comment(author, text)`: the HTML `<p><b>AUTHOR</b>: TEXT</p>` with both parts escaped with `html.escape`. And `find_notes(conn, owner, word)`: the ids of the owner's notes whose text contains `word` (case-insensitive, using `LIKE` with a `?` parameter), in id order.

In [ ]:
import html
import sqlite3
from pathlib import Path

def safe_path(base, name):
    ___

def render_comment(author, text):
    ___

def find_notes(conn, owner, word):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\nimport sqlite3\nfrom pathlib import Path\n\ndef test_paths():\n    \"no escaping the folder\"\n    os.makedirs(\"uploads/photos\", exist_ok=True)\n    base = Path(\"uploads\").resolve()\n    assert safe_path(\"uploads\", \"photos/cat.jpg\") == base / \"photos\" / \"cat.jpg\"\n    for bad in (\"../secrets.env\", \"photos/../../x\", \"/etc/passwd\"):\n        try:\n            safe_path(\"uploads\", bad)\n        except PermissionError as e:\n            assert str(e) == \"outside the folder\"\n        else:\n            raise AssertionError(bad)\n\ndef test_escaping_and_sql():\n    \"escape output, parameterise queries\"\n    assert render_comment(\"<Eve>\", \"<script>x()</script> & more\") == \"<p><b>&lt;Eve&gt;</b>: &lt;script&gt;x()&lt;/script&gt; &amp; more</p>\"\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE notes (id INTEGER PRIMARY KEY, owner TEXT, text TEXT)\")\n    conn.executemany(\"INSERT INTO notes (owner, text) VALUES (?, ?)\",\n                     [(\"ada\", \"Buy MILK\"), (\"bo\", \"milk run\"), (\"ada\", \"call Bo\"), (\"ada\", \"milkshake\")])\n    assert find_notes(conn, \"ada\", \"milk\") == [1, 4]\n    assert find_notes(conn, \"ada\", \"' OR '1'='1\") == []\n    assert \"?\" in source and \"f\\\"SELECT\" not in source and \"f'SELECT\" not in source")

### Exercise: A command-line tool

Write `main(argv, store)` for a tool called `notes`, using `argparse` with two subcommands: `add TEXT [--tag TAG]...` appends `{"text": TEXT, "tags": [...]}` to the list `store`, prints `added note N` (its number, from 1) and returns 0; `list [--tag TAG]` prints each note (all, or only those with that tag) as `N: TEXT`, or `no notes`, and returns 0. If the arguments are invalid, argparse raises `SystemExit`: catch it and return 2.

In [ ]:
import argparse

def main(argv, store):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import contextlib\nimport io\n\ndef _run(argv, store):\n    out = io.StringIO()\n    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(io.StringIO()):\n        code = main(argv, store)\n    return code, out.getvalue()\n\ndef test_cli():\n    \"subcommands and exit codes\"\n    store = []\n    assert _run([\"add\", \"buy milk\", \"--tag\", \"home\"], store) == (0, \"added note 1\\n\")\n    assert _run([\"add\", \"file report\", \"--tag\", \"work\", \"--tag\", \"urgent\"], store) == (0, \"added note 2\\n\")\n    assert store[1] == {\"text\": \"file report\", \"tags\": [\"work\", \"urgent\"]}\n    assert _run([\"list\"], store) == (0, \"1: buy milk\\n2: file report\\n\")\n    assert _run([\"list\", \"--tag\", \"home\"], store) == (0, \"1: buy milk\\n\")\n    assert _run([\"list\", \"--tag\", \"garden\"], store) == (0, \"no notes\\n\")\n    assert _run([\"remove\", \"1\"], store)[0] == 2 and _run([], store)[0] == 2")

### Bug hunt: A comment that runs

User text goes into the page without escaping, so a comment can run script in every reader's browser. Escape it so the program prints `<p>&lt;script&gt;alert(1)&lt;/script&gt;</p>`.

In [ ]:
import html

comment = "<script>alert(1)</script>"
print(f"<p>{comment}</p>")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"escape on output\"\n    expect_output(\"<p>&lt;script&gt;alert(1)&lt;/script&gt;</p>\")")

### Bug hunt: Downloading the password file

The check looks for `..` only at the very start of the name, so `docs/../../secrets.env` slips through: it prints `allowed`. Resolve the path and check it's still inside the folder, so it prints `blocked`.

In [ ]:
from pathlib import Path

base = Path("public").resolve()
name = "docs/../../secrets.env"
if name.startswith(".."):
    print("blocked")
else:
    print("allowed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"resolve, then check\"\n    expect_output(\"blocked\")")

### Bug hunt: A body one byte short

`Content-Length` counts **bytes**, but this counts characters, and "é" takes two bytes in UTF-8: it prints `Content-Length: 4`, and the client would cut the body short. Make it print `Content-Length: 5`.

In [ ]:
body = "café"
print(f"Content-Length: {len(body)}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"bytes, not characters\"\n    expect_output(\"Content-Length: 5\")")

### Puzzle: Handling one request

Arrange a request handler that parses the request, checks the token, routes, and turns any unexpected error into a 500 without leaking details. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status for "you're signed in, but this note isn't yours"
q = ___   # the status for "no token was sent"
r = ___   # the status for a successful DELETE with nothing to return
s = ___   # True or False: PUT is idempotent
t = ___   # the header that must come with a 405 response

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 403 and q == 401 and r == 204, \"p, q and r\"\n    assert s is True and t.lower() == \"allow\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a small JSON API

Write `NotesAPI(secret, clock)`, a JSON API using the given `Router`, `make_token` and `read_token`. `handle(method, path, headers, body)` returns `(status, data)` (`data` is a dict, or `None` for 204). Header names are case-insensitive. Endpoints:

- `POST /login` with JSON `{"user": NAME}` gives `200` and `{"token": ...}` (a token lasting 3600 seconds). (A real API would check a password; this one trusts the name.)
- `GET /notes`: the signed-in user's notes as `{"notes": [{"id", "text"}, ...]}` in id order.
- `POST /notes` with `{"text": ...}`: creates a note (ids from 1, shared by all users), giving `201` and the note.
- `GET /notes/<int:id>` and `DELETE /notes/<int:id>`: `404` if there's no such note, `403` if it belongs to someone else, else `200` with the note or `204`.

Every route except `/login` needs `Authorization: Bearer TOKEN`: missing or invalid gives `401` with `{"error": "sign in first"}`. A body that isn't valid JSON (or is missing a required string field) gives `400` with `{"error": "bad request"}`. An unknown route is `404` and a wrong method `405` with `{"error": "method not allowed", "allow": "..."}`. Any other exception becomes `500` with `{"error": "something went wrong"}`.

In [ ]:
import base64
import hashlib
import hmac
import json

class Router:
    def __init__(self):
        self.routes = []
    def add(self, method, pattern, handler):
        self.routes.append((method, pattern.strip("/").split("/"), handler))
    def match(self, method, path):
        segments = path.strip("/").split("/")
        allowed = set()
        for m, parts, handler in self.routes:
            if len(parts) != len(segments):
                continue
            params, ok = {}, True
            for part, seg in zip(parts, segments):
                if part.startswith("<int:"):
                    ok = seg.isdigit()
                    if ok:
                        params[part[5:-1]] = int(seg)
                elif part != seg:
                    ok = False
                if not ok:
                    break
            if ok:
                if m == method:
                    return 200, handler, params
                allowed.add(m)
        return (405, None, {"allow": ", ".join(sorted(allowed))}) if allowed else (404, None, {})

def _sign(data, secret):
    return hmac.new(secret, data.encode(), hashlib.sha256).hexdigest()

def make_token(user, secret, now, ttl):
    data = base64.urlsafe_b64encode(json.dumps({"user": user, "exp": now + ttl}).encode()).decode().rstrip("=")
    return f"{data}.{_sign(data, secret)}"

def read_token(token, secret, now):
    parts = token.split(".")
    if len(parts) != 2:
        raise ValueError("malformed token")
    data, sig = parts
    if not hmac.compare_digest(sig, _sign(data, secret)):
        raise ValueError("bad signature")
    payload = json.loads(base64.urlsafe_b64decode(data + "=" * (-len(data) % 4)))
    if now >= payload["exp"]:
        raise ValueError("expired")
    return payload["user"]

class NotesAPI:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SECRET = b\"test-\" + b\"api-\" + b\"key\"\n\nclass Clock:\n    def __init__(self):\n        self.t = 1000\n    def __call__(self):\n        return self.t\n\ndef _api():\n    clock = Clock()\n    return NotesAPI(SECRET, clock), clock\n\ndef _login(api, user):\n    status, data = api.handle(\"POST\", \"/login\", {}, '{\"user\": \"%s\"}' % user)\n    assert status == 200\n    return {\"authorization\": \"Bearer \" + data[\"token\"]}\n\ndef test_notes():\n    \"create, list, show, delete\"\n    api, clock = _api()\n    ada = _login(api, \"ada\")\n    assert api.handle(\"POST\", \"/notes\", ada, '{\"text\": \"buy milk\"}') == (201, {\"id\": 1, \"text\": \"buy milk\"})\n    assert api.handle(\"POST\", \"/notes\", {\"Authorization\": ada[\"authorization\"]}, '{\"text\": \"call Bo\"}')[0] == 201\n    assert api.handle(\"GET\", \"/notes\", ada, \"\") == (200, {\"notes\": [{\"id\": 1, \"text\": \"buy milk\"}, {\"id\": 2, \"text\": \"call Bo\"}]})\n    assert api.handle(\"GET\", \"/notes/2\", ada, \"\") == (200, {\"id\": 2, \"text\": \"call Bo\"})\n    assert api.handle(\"DELETE\", \"/notes/1\", ada, \"\") == (204, None)\n    assert api.handle(\"GET\", \"/notes/1\", ada, \"\")[0] == 404\n\ndef test_auth():\n    \"401 and 403\"\n    api, clock = _api()\n    ada, bo = _login(api, \"ada\"), _login(api, \"bo\")\n    api.handle(\"POST\", \"/notes\", ada, '{\"text\": \"private\"}')\n    assert api.handle(\"GET\", \"/notes/1\", bo, \"\")[0] == 403 and api.handle(\"DELETE\", \"/notes/1\", bo, \"\")[0] == 403\n    assert api.handle(\"GET\", \"/notes\", bo, \"\") == (200, {\"notes\": []})\n    assert api.handle(\"GET\", \"/notes\", {}, \"\") == (401, {\"error\": \"sign in first\"})\n    assert api.handle(\"GET\", \"/notes\", {\"authorization\": \"Bearer nonsense\"}, \"\")[0] == 401\n    clock.t += 3600\n    assert api.handle(\"GET\", \"/notes\", ada, \"\")[0] == 401, \"tokens expire\"\n\ndef test_errors():\n    \"400, 404, 405 and 500\"\n    api, clock = _api()\n    ada = _login(api, \"ada\")\n    assert api.handle(\"POST\", \"/notes\", ada, \"{not json\") == (400, {\"error\": \"bad request\"})\n    assert api.handle(\"POST\", \"/notes\", ada, '{\"words\": \"x\"}') == (400, {\"error\": \"bad request\"})\n    assert api.handle(\"POST\", \"/login\", {}, '{\"user\": 7}')[0] == 400\n    assert api.handle(\"GET\", \"/nowhere\", ada, \"\")[0] == 404\n    assert api.handle(\"PUT\", \"/notes/1\", ada, \"\") == (405, {\"error\": \"method not allowed\", \"allow\": \"DELETE, GET\"})\n    api.notes = None                      # break the storage on purpose\n    assert api.handle(\"GET\", \"/notes\", ada, \"\") == (500, {\"error\": \"something went wrong\"})")

## How to review

As yesterday: read the summary, predict each example **before** running it, then do the exercises without looking back, noting every topic you had to look up.

An HTTP request is just text: a request line, headers, a blank line, then the body. Everything a framework gives you (`request.method`, `request.json`) starts as this. If any part surprised you, start with Week 76.

## HTTP

- A **request** (Week 76) is `METHOD target VERSION`, then `Name: value` headers, a blank line (`\r\n\r\n`) and an optional body. Header names are **case-insensitive**. The target may carry a **query string** after `?` (`urllib.parse.parse_qs` splits it).
- A **response** is `HTTP/1.1 STATUS REASON`, headers, a blank line and the body. `Content-Length` is the body's length in **bytes**, not characters.
- **Status codes** say what happened: `200 OK`, `201 Created` (with a `Location` header for the new thing), `204 No Content`; `400` (malformed request), `401` (not authenticated: who are you?), `403` (authenticated but not allowed), `404`, `405 Method Not Allowed` (with an `Allow` header), `409 Conflict`, `422` (well-formed but invalid data); `500` (the server's fault); `429 Too Many Requests` and `503` with `Retry-After` when overloaded.
- **Methods**: `GET` reads and is **safe**; `PUT` and `DELETE` are **idempotent** (doing them twice has the same effect as once); `POST` creates and is neither, which is why payment APIs take idempotency keys (Week 77).

## APIs and frameworks

- **REST design** (Week 77): nouns for resources (`/notes`, `/notes/7`), methods for actions, plural collections, status codes for outcomes, JSON in and out, consistent error bodies (`{"error": "..."}`), pagination for long lists, and versioning (`/v1/...`) when the shape must change.
- A **framework** (Flask, FastAPI, or the course's miniflask) maps routes such as `/notes/<int:id>` to handler functions, converts path parts, parses JSON, and turns return values into responses. Underneath, it's a **router** (a list of patterns) and a loop that turns exceptions into error responses.
- **Authentication**: passwords are stored only as slow, salted hashes; after logging in, the client carries a **token**. A signed token is `payload.signature`, where the signature is an **HMAC** of the payload with a server secret: anyone can read the payload, but nobody without the secret can change it unnoticed. Check signatures with `hmac.compare_digest` (constant time), and give tokens an **expiry**.
- **Calling** APIs: set timeouts, check status codes, retry only safe or idempotent requests, with backoff (Week 82).

## Security

- **Injection** (Week 79): never build SQL, shell commands or file paths by pasting in user text. Use query **parameters**, argument lists instead of shell strings, and checked paths.
- **Cross-site scripting (XSS)**: user text shown in a page must be **escaped** (`html.escape`), so `<script>` is displayed rather than run. Templates escape by default; don't turn it off.
- **Path traversal**: a requested file name like `../../secrets.env` must not escape the folder it's meant for: resolve the full path and check it's still inside (`Path.is_relative_to`).
- **Secrets** come from the environment or a secrets manager, never the source code or the repository; `secrets.token_urlsafe` makes random tokens; compare them in constant time.
- **Dependencies** are code you run: pin them, update them, and audit them for known vulnerabilities.
- **Least privilege** and **defence in depth**: each part gets only the access it needs, and no single check is the only thing standing between an attacker and the data.

## Automation

- **Command-line tools** (Week 78) use `argparse`: positional arguments, `--options` with types and defaults, `store_true` flags, and **subcommands** (`add`, `list`). Write `main(argv)` so tests can call it with a list, and **return an exit code** (0 for success, non-zero for failure); `sys.exit(main(sys.argv[1:]))` at the bottom.
- **Files**: `pathlib` for paths, `glob` for patterns, writing to a temporary file and renaming it so readers never see half a file; **scheduling** with cron or a task scheduler, with logs and locks so runs don't overlap.
- **Scraping, responsibly**: read `robots.txt` and the site's terms, identify your client, go slowly, and cache.

## Classic mistakes

- Treating header names as case-sensitive; counting `Content-Length` in characters.
- `401` and `403` the wrong way round; `200` with an error message in the body; `500`s that leak a stack trace to the client.
- Comparing signatures with `==`; tokens that never expire; secrets committed to the repository.
- f-strings in SQL or shell commands; unescaped user text in pages; joining user-supplied paths without checking where they lead.
- Command-line tools that print errors but exit with `0`, so scripts calling them think everything worked.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the helpdesk project from Week 80 again and find one place where each of today's defences appears (parameters, escaping, path checks, signed tokens, secrets from the environment). If you can't find one, that's a gap worth noting. Tomorrow reviews **design, data and machine learning**, the last block before the practice exam.